# NASA Trend Detector — Data Access Tests

Testing API access to all three data sources:
1. **GES DISC** (earthaccess) — GLDAS, GPM, OMI
2. **AppEEARS** (REST API) — MODIS LST, NDVI, Burned Area
3. **LAADS DAAC** (blackmarblepy) — VIIRS Black Marble

Test region: **Delhi, India** — good for all phenomena (water stress, urban growth, air pollution, etc.)

In [37]:
# Common imports
import earthaccess
import xarray as xr
import pandas as pd
import numpy as np
import requests
import json
import time
import os
from datetime import datetime
from dotenv import load_dotenv

load_dotenv(override=True)

EARTHDATA_USER = os.getenv("EARTHDATA_USER")
EARTHDATA_PASS = os.getenv("EARTHDATA_PASS")
NASA_BEARER_TOKEN = os.getenv("NASA_BEARER_TOKEN")

# Test region: Delhi bounding box (west, south, east, north)
BBOX = (76.8, 28.4, 77.4, 28.9)
DATE_START = "2018-01-01"
DATE_END = "2023-12-31"

## 0. Earthdata Login

One account for all NASA data. Run once — saves credentials to `~/.netrc`.

In [38]:
auth = earthaccess.login(strategy="environment")
print("Logged in:", auth.authenticated)

Logged in: True


---
## 1. GES DISC — earthaccess

### 1.1 GLDAS (Groundwater Storage)

In [39]:
results_gldas = earthaccess.search_data(
    short_name="GLDAS_CLSM025_DA1_D",
    version="2.2",
    temporal=(DATE_START, DATE_END),
    bounding_box=BBOX,
    count=5  # just 5 granules for testing
)
print(f"GLDAS: found {len(results_gldas)} granules")
if results_gldas:
    print("First granule:", results_gldas[0])

GLDAS: found 5 granules
First granule: Collection: {'ShortName': 'GLDAS_CLSM025_DA1_D', 'Version': '2.2'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'WestBoundingCoordinate': -180.0, 'EastBoundingCoordinate': 180.0, 'NorthBoundingCoordinate': 90.0, 'SouthBoundingCoordinate': -60.0}]}}}
Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2018-01-01T00:00:00.000Z', 'EndingDateTime': '2018-01-01T23:59:59.000Z'}}
Size(MB): 19.588293075561523
Data: ['https://data.gesdisc.earthdata.nasa.gov/data/GLDAS/GLDAS_CLSM025_DA1_D.2.2/2018/01/GLDAS_CLSM025_DA1_D.A20180101.022.nc4']


In [40]:
# Try opening one granule
if results_gldas:
    files = earthaccess.open(results_gldas[:1])
    ds = xr.open_dataset(files[0])
    print("Variables:", list(ds.data_vars))
    print()
    if "GWS_tavg" in ds.data_vars:
        print("GWS_tavg:")
        print(ds["GWS_tavg"])
    ds.close()

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Variables: ['time_bnds', 'Swnet_tavg', 'Lwnet_tavg', 'Qle_tavg', 'Qh_tavg', 'Qg_tavg', 'Evap_tavg', 'Qs_tavg', 'Qsb_tavg', 'Qsm_tavg', 'SnowT_tavg', 'AvgSurfT_tavg', 'SWE_tavg', 'SnowDepth_tavg', 'SoilMoist_S_tavg', 'SoilMoist_RZ_tavg', 'SoilMoist_P_tavg', 'ECanop_tavg', 'TVeg_tavg', 'ESoil_tavg', 'CanopInt_tavg', 'EvapSnow_tavg', 'ACond_tavg', 'TWS_tavg', 'GWS_tavg']

GWS_tavg:
<xarray.DataArray 'GWS_tavg' (time: 1, lat: 600, lon: 1440)> Size: 3MB
[864000 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 8B 2018-01-01
  * lat      (lat) float32 2kB -59.88 -59.62 -59.38 -59.12 ... 89.38 89.62 89.88
  * lon      (lon) float32 6kB -179.9 -179.6 -179.4 -179.1 ... 179.4 179.6 179.9
Attributes:
    units:          mm
    standard_name:  Ground Water Storage
    long_name:      ground_water_storage
    cell_methods:   time: mean
    vmin:           77.03008
    vmax:           3583.7908


### 1.2 GPM IMERG (Precipitation)

In [41]:
results_gpm = earthaccess.search_data(
    short_name="GPM_3IMERGM",
    version="07",
    temporal=(DATE_START, DATE_END),
    bounding_box=BBOX,
    count=5
)
print(f"GPM: found {len(results_gpm)} granules")
if results_gpm:
    print("First granule:", results_gpm[0])

GPM: found 5 granules
First granule: Collection: {'ShortName': 'GPM_3IMERGM', 'Version': '07'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'WestBoundingCoordinate': -180.0, 'EastBoundingCoordinate': 180.0, 'NorthBoundingCoordinate': 90.0, 'SouthBoundingCoordinate': -90.0}]}}}
Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2018-01-01T00:00:00.000Z', 'EndingDateTime': '2018-01-31T23:59:59.999Z'}}
Size(MB): 17.071714401245117
Data: ['https://data.gesdisc.earthdata.nasa.gov/data/GPM_L3/GPM_3IMERGM.07/2018/3B-MO.MS.MRG.3IMERG.20180101-S000000-E235959.01.V07B.HDF5']


In [42]:
if results_gpm:
    files = earthaccess.open(results_gpm[:1])
    ds = xr.open_dataset(files[0], group="Grid", decode_times=False)
    print("Variables:", list(ds.data_vars))
    print()
    print(ds)
    ds.close()

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Variables: ['time_bnds', 'lon_bnds', 'lat_bnds', 'precipitation', 'randomError', 'gaugeRelativeWeighting', 'probabilityLiquidPrecipitation', 'precipitationQualityIndex']

<xarray.Dataset> Size: 130MB
Dimensions:                         (time: 1, nv: 2, lon: 3600, lonv: 2,
                                     lat: 1800, latv: 2)
Coordinates:
  * time                            (time) int32 4B 1198800000
  * lon                             (lon) float32 14kB -179.9 -179.9 ... 179.9
  * lat                             (lat) float32 7kB -89.95 -89.85 ... 89.95
Dimensions without coordinates: nv, lonv, latv
Data variables:
    time_bnds                       (time, nv) int32 8B ...
    lon_bnds                        (lon, lonv) float32 29kB ...
    lat_bnds                        (lat, latv) float32 14kB ...
    precipitation                   (time, lon, lat) float32 26MB ...
    randomError                     (time, lon, lat) float32 26MB ...
    gaugeRelativeWeighting          (time, l

### 1.3 OMI NO₂

In [43]:
results_omi = earthaccess.search_data(
    short_name="OMNO2d",
    version="003",
    temporal=(DATE_START, DATE_END),
    bounding_box=BBOX,
    count=5
)
print(f"OMI NO2: found {len(results_omi)} granules")
if results_omi:
    print("First granule:", results_omi[0])

OMI NO2: found 5 granules
First granule: Collection: {'ShortName': 'OMNO2d', 'Version': '003'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'WestBoundingCoordinate': -180.0, 'EastBoundingCoordinate': 180.0, 'NorthBoundingCoordinate': 90.0, 'SouthBoundingCoordinate': -90.0}]}}}
Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2018-01-01T00:00:00.000Z', 'EndingDateTime': '2018-01-01T23:59:59.000Z'}}
Size(MB): 10.6558179855347
Data: ['https://data.gesdisc.earthdata.nasa.gov/data/Aura_OMI_Level3/OMNO2d.003/2018/OMI-Aura_L3-OMNO2d_2018m0101_v003-2019m1123t055315.he5']


In [44]:
if results_omi:
    files = earthaccess.open(results_omi[:1])
    ds = xr.open_dataset(files[0], group="HDFEOS/GRIDS/ColumnAmountNO2/Data Fields")
    print("Variables:", list(ds.data_vars))
    print()
    print(ds)
    ds.close()

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Variables: ['ColumnAmountNO2', 'ColumnAmountNO2CloudScreened', 'ColumnAmountNO2Trop', 'ColumnAmountNO2TropCloudScreened', 'Weight']

<xarray.Dataset> Size: 21MB
Dimensions:                           (phony_dim_0: 720, phony_dim_1: 1440)
Dimensions without coordinates: phony_dim_0, phony_dim_1
Data variables:
    ColumnAmountNO2                   (phony_dim_0, phony_dim_1) float32 4MB ...
    ColumnAmountNO2CloudScreened      (phony_dim_0, phony_dim_1) float32 4MB ...
    ColumnAmountNO2Trop               (phony_dim_0, phony_dim_1) float32 4MB ...
    ColumnAmountNO2TropCloudScreened  (phony_dim_0, phony_dim_1) float32 4MB ...
    Weight                            (phony_dim_0, phony_dim_1) float32 4MB ...


/tmp/ipykernel_93380/3064584005.py:3: UserWarning: The 'phony_dims' kwarg now defaults to 'access'. Previously 'phony_dims=None' would raise an error. For full netcdf equivalence please use phony_dims='sort'.
  ds = xr.open_dataset(files[0], group="HDFEOS/GRIDS/ColumnAmountNO2/Data Fields")


---
## 2. AppEEARS — REST API

Async task-based workflow: submit → wait → download.

### 2.0 Auth

In [45]:
APPEEARS_API = "https://appeears.earthdatacloud.nasa.gov/api"

In [46]:
login_resp = requests.post(
    f"{APPEEARS_API}/login",
    auth=(EARTHDATA_USER, EARTHDATA_PASS)
)
print("Status:", login_resp.status_code)

if login_resp.status_code == 200:
    token = login_resp.json()["token"]
    appeears_headers = {"Authorization": f"Bearer {token}"}
    print("AppEEARS auth OK")
else:
    print("Auth failed:", login_resp.text)

Status: 200
AppEEARS auth OK


### 2.1 Check available products

In [47]:
# Check that our products exist
# MOD11C3/MOD13C2 (CMG) are NOT in AppEEARS — use tiled products instead:
#   MOD13A3.061 = NDVI monthly 1km
#   MOD11A2.061 = LST 8-day 1km
products_resp = requests.get(f"{APPEEARS_API}/product").json()

our_products = ["MOD11A2.061", "MOD13A3.061", "MCD64A1.061"]
for prod_id in our_products:
    found = any(p["ProductAndVersion"] == prod_id for p in products_resp)
    print(f"{prod_id}: {'FOUND' if found else 'NOT FOUND'}")

MOD11A2.061: FOUND
MOD13A3.061: FOUND
MCD64A1.061: FOUND


In [48]:
# List layers for each product
for prod_id in our_products:
    layers_resp = requests.get(f"{APPEEARS_API}/product/{prod_id}").json()
    print(f"\n=== {prod_id} ===")
    for layer_name in list(layers_resp.keys())[:10]:
        print(f"  {layer_name}")


=== MOD11A2.061 ===
  Clear_sky_days
  Clear_sky_nights
  Day_view_angl
  Day_view_time
  Emis_31
  Emis_32
  LST_Day_1km
  LST_Night_1km
  Night_view_angl
  Night_view_time

=== MOD13A3.061 ===
  _1_km_monthly_EVI
  _1_km_monthly_MIR_reflectance
  _1_km_monthly_NDVI
  _1_km_monthly_NIR_reflectance
  _1_km_monthly_VI_Quality
  _1_km_monthly_blue_reflectance
  _1_km_monthly_pixel_reliability
  _1_km_monthly_red_reflectance
  _1_km_monthly_relative_azimuth_angle
  _1_km_monthly_sun_zenith_angle

=== MCD64A1.061 ===
  Burn_Date
  Burn_Date_Uncertainty
  First_Day
  Last_Day
  QA


### 2.2 Submit a test task (NDVI)

In [49]:
# Small test: 1 year of monthly NDVI for Delhi area
task_payload = {
    "task_type": "area",
    "task_name": "test_ndvi_delhi",
    "params": {
        "dates": [
            {"startDate": "01-01-2022", "endDate": "12-31-2022"}
        ],
        "layers": [
            {"product": "MOD13A3.061", "layer": "_1_km_monthly_NDVI"}
        ],
        "output": {
            "format": {"type": "geotiff"},
            "projection": "geographic"
        },
        "geo": {
            "type": "FeatureCollection",
            "features": [{
                "type": "Feature",
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [[
                        [76.8, 28.4],
                        [77.4, 28.4],
                        [77.4, 28.9],
                        [76.8, 28.9],
                        [76.8, 28.4]
                    ]]
                },
                "properties": {}
            }]
        }
    }
}

task_resp = requests.post(
    f"{APPEEARS_API}/task",
    json=task_payload,
    headers=appeears_headers
)
print("Status:", task_resp.status_code)

if task_resp.status_code == 202:
    task_id = task_resp.json()["task_id"]
    print(f"Task submitted! ID: {task_id}")
else:
    print("Error:", task_resp.text)

Status: 202
Task submitted! ID: a94228d8-98ca-411d-9d66-d3274a8134c5


In [50]:
# Poll for task completion
for i in range(20):
    status_resp = requests.get(
        f"{APPEEARS_API}/task/{task_id}",
        headers=appeears_headers
    ).json()
    status = status_resp["status"]
    print(f"[{i+1}] Status: {status}")
    if status == "done":
        print("Task complete!")
        break
    elif status == "error":
        print("Task failed:", status_resp)
        break
    time.sleep(30)

[1] Status: pending
[2] Status: processing
[3] Status: processing
[4] Status: processing
[5] Status: done
Task complete!


In [51]:
# List output files
bundle = requests.get(
    f"{APPEEARS_API}/bundle/{task_id}",
    headers=appeears_headers
).json()

print(f"Files in bundle: {len(bundle['files'])}")
for f in bundle["files"][:5]:
    print(f"  {f['file_name']}  ({f['file_size']} bytes)")

Files in bundle: 31
  MOD13A3.061_2022001_to_2022365/MOD13A3.061__1_km_monthly_NDVI_doy2022001000000_aid0001.tif  (13089 bytes)
  MOD13A3.061_2022001_to_2022365/MOD13A3.061__1_km_monthly_NDVI_doy2022032000000_aid0001.tif  (13185 bytes)
  MOD13A3.061_2022001_to_2022365/MOD13A3.061__1_km_monthly_NDVI_doy2022060000000_aid0001.tif  (12999 bytes)
  MOD13A3.061_2022001_to_2022365/MOD13A3.061__1_km_monthly_NDVI_doy2022091000000_aid0001.tif  (12527 bytes)
  MOD13A3.061_2022001_to_2022365/MOD13A3.061__1_km_monthly_NDVI_doy2022121000000_aid0001.tif  (12458 bytes)


---
## 3. LAADS DAAC — VIIRS Black Marble

Using `blackmarblepy` package.

In [52]:
import geopandas as gpd
from shapely.geometry import box
from blackmarble.raster import bm_raster

# Create GeoDataFrame for Delhi
delhi_geom = box(76.8, 28.4, 77.4, 28.9)
gdf = gpd.GeoDataFrame(geometry=[delhi_geom], crs="EPSG:4326")
print("Region:")
print(gdf)

Region:
                                            geometry
0  POLYGON ((77.4 28.4, 77.4 28.9, 76.8 28.9, 76....


In [53]:
# Token already loaded from .env

In [54]:
# Test: get yearly nighttime lights for 1 year only (less data)
ntl = bm_raster(
    gdf,
    product_id="VNP46A4",
    date_range=pd.date_range("2022-01-01", "2022-12-31", freq="YS"),
    token=NASA_BEARER_TOKEN
)

print(type(ntl))
print(ntl)

QUEUEING TASKS | Downloading (146.0 MB)...:   0%|          | 0/1 [00:00<?, ?file/s]

PROCESSING TASKS | Downloading (146.0 MB)...:   0%|          | 0/1 [00:00<?, ?file/s]

  0%|          | 0.00/146M [00:00<?, ?B/s]

COLLECTING RESULTS | Downloading (146.0 MB)...:   0%|          | 0/1 [00:00<?, ?file/s]

COLLATING TILES | Processing...:   0%|          | 0/1 [00:00<?, ?date/s]

<class 'xarray.core.dataset.Dataset'>
<xarray.Dataset> Size: 140kB
Dimensions:                        (x: 144, y: 120, time: 1)
Coordinates:
  * x                              (x) float64 1kB 76.8 76.81 ... 77.39 77.4
  * y                              (y) float64 960B 28.9 28.89 ... 28.41 28.4
  * time                           (time) datetime64[ns] 8B 2022-01-01
Data variables:
    NearNadir_Composite_Snow_Free  (time, y, x) float64 138kB 1.022 ... 2.771
Attributes: (12/53)
    AlgorithmType:                     b'SCI'
    AlgorithmVersion:                  b'NPP_PR46A3 2.0.0'
    Conventions:                       b'CF-1.6'
    creator_email:                     b'modis-ops@lists.nasa.gov'
    creator_name:                      b'VIIRS Land SIPS Processing Group'
    creator_url:                       b'https://ladsweb.modaps.eosdis.nasa.gov'
    ...                                ...
    VerticalTileNumber:                b'06'
    WestBoundingCoord:                 70.0
    AREA_O

---
## Summary

| Source | Dataset | Works? | Notes |
|--------|---------|--------|-------|
| earthaccess | GLDAS | ✅ | GWS_tavg, TWS_tavg. Grid 600x1440 (0.25°). Opens directly with xarray |
| earthaccess | GPM | ✅ | precipitation. Grid 3600x1800 (0.1°). Requires `group="Grid"`, `decode_times=False` |
| earthaccess | OMI NO₂ | ✅ | ColumnAmountNO2TropCloudScreened. 720x1440. Requires `group="HDFEOS/GRIDS/ColumnAmountNO2/Data Fields"` |
| AppEEARS | MODIS NDVI (MOD13A3.061) | ✅ | Layer: `_1_km_monthly_NDVI`. Async task ~2.5 min. 31 GeoTIFF files for 1 year |
| AppEEARS | MODIS LST (MOD11A2.061) | ✅ | Layers: `LST_Day_1km`, `LST_Night_1km`. Product confirmed available |
| AppEEARS | Burned Area (MCD64A1.061) | ✅ | Layers: `Burn_Date`, `Burn_Date_Uncertainty`. Product confirmed available |
| blackmarblepy | VIIRS NTL (VNP46A4) | ✅ | `bm_raster()` → xarray Dataset (144x120 for Delhi). ~146 MB/year download. Requires LAADS EULA acceptance |